In [0]:
dbutils.widgets.text("run_month", "2026-04", "Run month (YYYY-MM)")

run_month = dbutils.widgets.get("run_month")
print(f"Green Taxi run month: {run_month}")

In [0]:
source_dir = (
    "/Volumes/nyc_mobility/nyc_bronze/ftw-b12-de-r2/"
    "groups/week-08/group-c/landing/green_taxi"
)

file_path = f"{source_dir}/green_tripdata_{run_month}.parquet"
taxi_df = spark.read.parquet(file_path)

print("Rows:", taxi_df.count())
taxi_df.printSchema()
display(taxi_df.limit(5))

In [0]:
from pyspark.sql import functions as F

taxi_df.select(
    F.min("lpep_pickup_datetime").alias("earliest_pickup"),
    F.max("lpep_pickup_datetime").alias("latest_pickup"),
    F.sum(F.col("lpep_pickup_datetime").isNull().cast("int")).alias("missing_pickup"),
    F.sum(
        (
            F.date_format("lpep_pickup_datetime", "yyyy-MM") != run_month
        ).cast("int")
    ).alias("outside_run_month"),
).show()

In [0]:
taxi_df.groupBy(
    F.date_format("lpep_pickup_datetime", "yyyy-MM").alias("pickup_month")
).count().orderBy("pickup_month").show()

In [0]:
display(
    taxi_df
    .filter(F.date_format("lpep_pickup_datetime", "yyyy-MM") != run_month)
    .select("lpep_pickup_datetime", "lpep_dropoff_datetime", "trip_distance")
    .orderBy("lpep_pickup_datetime")
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS nyc_mobility.nyc_bronze.green_taxi_raw
""")
print("Bronze table is ready.")

In [0]:
import re

run_month = dbutils.widgets.get("run_month")

if not re.fullmatch(r"2026-(03|04|05)", run_month):
    raise ValueError("Run month must be 2026-03, 2026-04, or 2026-05")

source_dir = (
    "/Volumes/nyc_mobility/nyc_bronze/ftw-b12-de-r2/"
    "groups/week-08/group-c/landing/green_taxi"
)

result = spark.sql(f"""
    COPY INTO nyc_mobility.nyc_bronze.green_taxi_raw
    FROM '{source_dir}'
    FILEFORMAT = PARQUET
    FILES = ('green_tripdata_{run_month}.parquet')
    COPY_OPTIONS ('mergeSchema' = 'true')
""")

display(result)

In [0]:
spark.sql("""
ALTER TABLE nyc_mobility.nyc_bronze.green_taxi_raw
SET TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported')
""")

In [0]:
spark.sql("""
SELECT COUNT(*) AS bronze_rows
FROM nyc_mobility.nyc_bronze.green_taxi_raw
""").show()

In [0]:
spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  COUNT_IF(lpep_pickup_datetime IS NULL) AS missing_pickup,
  COUNT_IF(
    lpep_pickup_datetime < TIMESTAMP_NTZ '2026-03-01 00:00:00'
    OR lpep_pickup_datetime >= TIMESTAMP_NTZ '2026-06-01 00:00:00'
  ) AS outside_project_period
FROM nyc_mobility.nyc_bronze.green_taxi_raw
""").show()

In [0]:
spark.sql("""
SELECT
  COUNT_IF(lpep_dropoff_datetime IS NULL) AS missing_dropoff,
  COUNT_IF(lpep_dropoff_datetime < lpep_pickup_datetime) AS dropoff_before_pickup,
  COUNT_IF(trip_distance IS NULL OR trip_distance <= 0) AS invalid_distance,
  COUNT_IF(PULocationID IS NULL OR DOLocationID IS NULL) AS missing_zone
FROM nyc_mobility.nyc_bronze.green_taxi_raw
""").show()

In [0]:
spark.sql("""
SELECT
  COUNT_IF(trip_distance IS NULL) AS missing_distance,
  COUNT_IF(trip_distance = 0) AS zero_distance,
  COUNT_IF(trip_distance < 0) AS negative_distance
FROM nyc_mobility.nyc_bronze.green_taxi_raw
""").show()

In [0]:
spark.sql("""
CREATE VIEW IF NOT EXISTS nyc_mobility.nyc_silver.vw_green_taxi_clean AS
SELECT
    lpep_pickup_datetime AS pickup_datetime,
    lpep_dropoff_datetime AS dropoff_datetime,
    PULocationID AS pickup_zone_id,
    DOLocationID AS dropoff_zone_id,
    passenger_count,
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount,
    trip_distance = 0 AS is_zero_distance
FROM nyc_mobility.nyc_bronze.green_taxi_raw
WHERE lpep_pickup_datetime >= TIMESTAMP_NTZ '2026-03-01 00:00:00'
  AND lpep_pickup_datetime < TIMESTAMP_NTZ '2026-06-01 00:00:00'
  AND lpep_dropoff_datetime >= lpep_pickup_datetime
""")

print("Silver view is ready.")

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS silver_rows,
    COUNT_IF(is_zero_distance) AS zero_distance_rows
FROM nyc_mobility.nyc_silver.vw_green_taxi_clean
""").show()

In [0]:
bronze_df = spark.table("nyc_mobility.nyc_bronze.green_taxi_raw")

duplicate_groups = (
    bronze_df.groupBy(*bronze_df.columns)
    .count()
    .filter("count > 1")
)

duplicate_groups.selectExpr(
    "count(*) AS duplicate_groups",
    "coalesce(sum(count - 1), 0) AS extra_rows"
).show()